In [ ]:
# ── Colab Setup (run this first) ──────────────────────────────────────────
!pip install openai tiktoken -q

import os
# Option 1: Paste your key directly
os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Notebook 04 — Fine-Tuning Demo (Bonus)
AutoRFP-AI: Show the Fine-Tuning Workflow

WHAT THIS DOES:
- Formats training data into OpenAI fine-tuning JSONL
- Validates format and estimates cost
- Shows the exact API calls to fine-tune
- Optionally runs a real fine-tuning job (~$1)

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

### Setup

In [ ]:
import os
import json
import tiktoken
from pathlib import Path
from openai import OpenAI

client = OpenAI()
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

with open(DATA_DIR / "past_responses.json") as f:
    knowledge_base = json.load(f)
print(f"Loaded {len(knowledge_base)} Q&A pairs")

### Format for fine-tuning

In [ ]:
SYSTEM_PROMPT = """You are a senior sales engineer at Acme Cloud Platform.
Draft professional, specific RFP responses with concrete details,
certifications, timelines, and metrics. Confident but not arrogant,
detailed but concise (80-200 words)."""


def format_for_fine_tuning(qa_pairs):
    formatted = []
    for qa in qa_pairs:
        formatted.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Draft an RFP response for: {qa['question']}"},
                {"role": "assistant", "content": qa["answer"]},
            ]
        })
    return formatted


training_data = format_for_fine_tuning(knowledge_base)
print(f"Formatted {len(training_data)} training examples")

# Preview
print("\n-- Sample entry --")
print(json.dumps(training_data[0], indent=2)[:400] + "...")

### Save as JSONL

In [ ]:
jsonl_path = DATA_DIR / "fine_tune_training.jsonl"
with open(jsonl_path, "w") as f:
    for entry in training_data:
        f.write(json.dumps(entry) + "\n")
print(f"Saved to {jsonl_path}")

### Validate and estimate cost

In [ ]:
enc = tiktoken.encoding_for_model("gpt-4o-mini")
total_tokens = 0
for entry in training_data:
    for msg in entry["messages"]:
        total_tokens += len(enc.encode(msg["content"]))
    total_tokens += 3  # per-message overhead

epochs = 3
training_tokens = total_tokens * epochs
cost = (training_tokens / 1000) * 0.0003

print(f"Training Data Stats:")
print(f"  Examples:         {len(training_data)}")
print(f"  Total tokens:     {total_tokens:,}")
print(f"  Epochs:           {epochs}")
print(f"  Training tokens:  {training_tokens:,}")
print(f"  Estimated cost:   ${cost:.2f}")

# Validate format
errors = []
for i, entry in enumerate(training_data):
    if "messages" not in entry:
        errors.append(f"Entry {i}: missing 'messages'")
    elif len(entry["messages"]) != 3:
        errors.append(f"Entry {i}: wrong message count")
    elif entry["messages"][0]["role"] != "system":
        errors.append(f"Entry {i}: first msg not system")

print(f"\nFormat: {'All valid' if not errors else f'{len(errors)} errors'}")
for e in errors:
    print(f"  {e}")

### Fine-tuning API code (reference)

In [ ]:
print("\n" + "=" * 50)
print("FINE-TUNING API WORKFLOW")
print("=" * 50)
print("""
# ── Step 1: Upload training file ──
upload = client.files.create(
    file=open("data/fine_tune_training.jsonl", "rb"),
    purpose="fine-tune"
)
file_id = upload.id

# ── Step 2: Create fine-tuning job ──
job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={"n_epochs": 3},
    suffix="autorfp"
)
print(f"Job: {job.id}, Status: {job.status}")

# ── Step 3: Monitor ──
import time
while True:
    job = client.fine_tuning.jobs.retrieve(job.id)
    print(f"Status: {job.status}")
    if job.status in ["succeeded", "failed", "cancelled"]:
        break
    time.sleep(60)

# ── Step 4: Use the model ──
ft_model = job.fine_tuned_model
# Then in agents.py, change: MODEL = ft_model
""")

### Optional — actually run it

In [ ]:
RUN_FINE_TUNING = False  # Set True to run (~$1)

if RUN_FINE_TUNING:
    print("Running fine-tuning with 10 examples...")

    mini_jsonl = DATA_DIR / "fine_tune_mini.jsonl"
    with open(mini_jsonl, "w") as f:
        for entry in training_data[:10]:
            f.write(json.dumps(entry) + "\n")

    upload = client.files.create(file=open(mini_jsonl, "rb"), purpose="fine-tune")
    print(f"  Uploaded: {upload.id}")

    job = client.fine_tuning.jobs.create(
        training_file=upload.id,
        model="gpt-4o-mini-2024-07-18",
        hyperparameters={"n_epochs": 3},
        suffix="autorfp-demo"
    )
    print(f"  Job: {job.id}")
    print(f"  Status: {job.status}")
    print(f"  Monitor: https://platform.openai.com/finetune/{job.id}")
else:
    print("Fine-tuning not executed (RUN_FINE_TUNING = False)")
    print("Set to True to run (~$1)")

### Few-shot vs Fine-tuned comparison

In [ ]:
print("\n" + "=" * 50)
print("FEW-SHOT vs FINE-TUNED")
print("=" * 50)
print("""
  Few-Shot (this demo)      Fine-Tuned (production)
  ─────────────────────     ──────────────────────
  Setup: minutes            Setup: hours
  Cost: free                Cost: $1-50+
  Voice match: ~80%         Voice match: ~95%
  Context usage: high       Context usage: low
  Best for: prototypes      Best for: production

  For a portfolio project, few-shot is the right call.
  For production, fine-tune on 200-500+ real Q&A pairs.
""")
print("Notebook 04 complete!")